# TradeSense · Component 03 · notebook 54: "what kind of abnormal behaviour does this resemble?" (CPU, no deep training)

**Goal.** TradeSense does not predict price and gives no buy/sell signal. It helps a trader judge: can I trust the current BTC/ETH behaviour, what makes it potentially abnormal, how is it evolving, how reliable is the evidence. This notebook serves *what makes it potentially abnormal*: for a flagged episode, does it **resemble** one of the six trained pattern families, or none (possible unseen pattern)?

Pre-registered in `protocol_g1_v1_pattern_resemblance.md` (put it in `research datasets/layer3_v3/`; its sha256 is frozen at the first run). Fit on TRAIN-3, choose C and reject thresholds on VAL-3 (known families only), report CAL-3 once. TEST-3, TEST-B and TGNN test days are never read; the two unseen families are never used to fit or tune.

Hypotheses (bars fixed before any result): **H-G1** ORACLE macro top-1 >= 0.50 (bootstrap lower > 0.33); **H-G2** ORACLE macro top-2 >= 0.75; **H-G3** JITTERED unseen reject >= 0.40 with known reject <= 0.15; **H-G5** JITTERED macro top-1 >= 0.40 and top-2 >= 0.65. Failed hypotheses are reported as failed and the matching dashboard output stays hidden.

In [ ]:
import os, sys, json, glob, re, hashlib, warnings, datetime, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
# frozen B2 protocol must be intact (same check as notebook 45)
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_final_set_v1.json']: assert (OUT/f).exists(), f'{f} missing'
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; importlib.reload(B)
# protocol G1: freeze its hash at the first run, then require it unchanged
g1 = L3/'protocol_g1_v1_pattern_resemblance.md'; g1h = L3/'protocol_g1_v1_pattern_resemblance.sha256'
assert g1.exists(), 'copy protocol_g1_v1_pattern_resemblance.md into ' + str(L3)
gh = hashlib.sha256(g1.read_text().encode()).hexdigest()
if not g1h.exists(): g1h.write_text(gh + '  protocol_g1_v1_pattern_resemblance.md\n'); say('protocol G1 frozen now, sha256 ' + gh[:16])
else: assert gh == g1h.read_text().split()[0], 'protocol G1 was changed after freezing'; say('protocol G1 intact, sha256 ' + gh[:16])
say('protocol B2 intact, sha256 ' + ph[:16] + '; library sha256 ' + sha_file(OUT/'b2_lib.py'))
FEATS = B.FEATS; FAMN = B.FAMS
assert len(FEATS) == 6 and 'momentum' in str(FAMN[7]).lower() and 'slow' in str(FAMN[8]).lower(), (FEATS, FAMN)
DF1 = json.load(open(OUT/'declared_final_set_v1.json')); MODE = DF1['models']['XGBoost v2']['normalisation']
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
N0 = {}
for s in SYMBOLS:
    a_ = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a_ = a_.set_index('datetime') if 'datetime' in a_.columns else a_
    a_.loc[a_.index[:15], 'volume_acceleration'] = 0.0; N0[s] = B.fit_n0(a_[FEATS].reset_index(drop=True).iloc[15:90720])
say(f'normalisation mode {MODE}; families: ' + str({k: v for k, v in FAMN.items()}))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
protocol G1 frozen now, sha256 6ffd7e485c67f253
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944
normalisation mode N0; families: {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy', 6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


## 1 · Splits

In [ ]:
# find the splits by name; TEST keys are never opened
names = {}
for s in SYMBOLS:
    found = sorted(re.findall(rf'l3v3_{s}_(.+)_events\.csv$', ' '.join(os.listdir(L3)).replace(' ', '\n'), flags=re.M))
    names[s] = found
say('splits found in layer3_v3: ' + str(names))
def pick(s, prefix):
    c = [n for n in names[s] if n.upper().startswith(prefix) and 'TEST' not in n.upper()]
    assert len(c) == 1, f'expected exactly one {prefix} split for {s}, found {c}: tell Claude the names'
    return c[0]
SPLIT = {s: {'TRAIN': pick(s, 'TRAIN'), 'VAL': pick(s, 'VAL'), 'CAL': pick(s, 'CAL')} for s in SYMBOLS}
say('using splits: ' + str(SPLIT)); assert not any('TEST' in v.upper() for d in SPLIT.values() for v in d.values())

splits found in layer3_v3: {'BTCUSDT': ['CAL', 'TEST', 'TRAIN', 'VAL'], 'ETHUSDT': ['CAL', 'TEST', 'TRAIN', 'VAL']}
using splits: {'BTCUSDT': {'TRAIN': 'TRAIN', 'VAL': 'VAL', 'CAL': 'CAL'}, 'ETHUSDT': {'TRAIN': 'TRAIN', 'VAL': 'VAL', 'CAL': 'CAL'}}


## 2 · Core logic

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import LedoitWolf
KNOWN = [1, 2, 3, 4, 5, 6]; UNSEEN = [7, 8]; CS = [0.01, 0.1, 1, 10]; BOOT = 2000

def signature(z, a, b):
    w = np.asarray(z[a:b], np.float64)
    if len(w) < 2 or not np.isfinite(w).all(): return None
    return np.concatenate([np.abs(w).max(0), w.mean(0), [np.log(len(w))]])

def jitter(uid, t0, dur):
    r = np.random.default_rng(1000003 * (uid + 1)); k = int(r.integers(0, min(5, dur // 3) + 1)); m = int(r.integers(0, 11))
    return t0 + k, t0 + dur + m

def macro(y, hit):
    return float(np.mean([hit[y == f].mean() for f in np.unique(y)]))

def topk_hits(P, classes, y, k):
    order = np.argsort(-P, axis=1)[:, :k]; return np.array([y[i] in classes[order[i]] for i in range(len(y))], float)

def fit_known(Xtr, ytr, Xva, yva):
    sc = StandardScaler().fit(Xtr); A = sc.transform(Xtr); V = sc.transform(Xva); best = None
    for C in CS:
        lr = LogisticRegression(C=C, max_iter=5000, class_weight='balanced').fit(A, ytr)
        s_ = macro(yva, topk_hits(lr.predict_proba(V), lr.classes_, yva, 1))
        if best is None or s_ > best[0] + 1e-12: best = (s_, C, lr)
    _, C, lr = best; cent = {f: A[ytr == f].mean(0) for f in KNOWN}
    resid = np.vstack([A[ytr == f] - cent[f] for f in KNOWN]); P = LedoitWolf().fit(resid).precision_
    return sc, lr, C, cent, P

def mdist(x, c, P):
    d = x - c; return float(np.sqrt(max(d @ P @ d, 0.0)))

def thresholds(sc, cent, P, Xva, yva):
    V = sc.transform(Xva); return {f: float(np.percentile([mdist(v, cent[f], P) for v in V[yva == f]], 90)) for f in KNOWN}

def predict(sc, lr, cent, P, thr, X):
    S = sc.transform(X); Pr = lr.predict_proba(S); top = lr.classes_[np.argmax(Pr, axis=1)]
    d = np.array([mdist(S[i], cent[top[i]], P) for i in range(len(S))]); rej = d > np.array([thr[t] for t in top]); return Pr, top, d, rej

def boot_lower(y, hit, seed=7):
    r = np.random.default_rng(seed); idx = np.arange(len(y)); out = []
    for _ in range(BOOT):
        s_ = r.choice(idx, len(idx), replace=True); out.append(macro(y[s_], hit[s_]))
    return float(np.percentile(out, 2.5))


## 3 · Tests and datasets

In [ ]:
def build(part, only_replica0=False):
    # returns dict variant -> (X, y, uid, sym) built from the event bars (ORACLE) or jittered bars (JITTERED)
    out = {'ORACLE': [[], [], [], []], 'JITTERED': [[], [], [], []]}; uid = 0
    for s in SYMBOLS:
        nm = SPLIT[s][part]; f = pd.read_parquet(L3/f'l3v3_{s}_{nm}.parquet'); e = pd.read_csv(L3/f'l3v3_{s}_{nm}_events.csv')
        if 'replica' not in f.columns: f['replica'] = 0
        if 'replica' not in e.columns: e['replica'] = 0
        for rp in sorted(f.replica.unique()):
            if only_replica0 and rp != 0: continue
            g = f[f.replica == rp].reset_index(drop=True); ev = e[e.replica == rp]
            z = B.normalise(g[FEATS], MODE, N0[s])
            for r in ev.itertuples():
                uid += 1; fam = int(r.family)
                for var, (a, b) in {'ORACLE': (int(r.t0), int(r.t0 + r.dur)), 'JITTERED': jitter(uid, int(r.t0), int(r.dur))}.items():
                    sg = signature(z, a, min(b, len(z)))
                    if sg is None: continue
                    o = out[var]; o[0].append(sg); o[1].append(fam); o[2].append(uid); o[3].append(s)
    return {k: (np.array(v[0]), np.array(v[1]), np.array(v[2]), np.array(v[3])) for k, v in out.items()}
# unit tests of the logic
z_ = np.random.default_rng(0).normal(size=(60, 6)); assert signature(z_, 10, 20).shape == (13,) and signature(z_, 10, 11) is None
z2 = z_.copy(); z2[30:] += 5; assert np.allclose(signature(z_, 10, 20), signature(z2, 10, 20)); say('PASS T01 signature uses only the episode bars (later bars cannot change it)')
assert jitter(3, 100, 12) == jitter(3, 100, 12) and jitter(3, 100, 12)[0] >= 100; say('PASS T02 jitter is deterministic per event and never starts before the event')
assert not any('TEST' in str(p).upper() for p in [L3/f'l3v3_{s}_{SPLIT[s][k]}' for s in SYMBOLS for k in SPLIT[s]]); say('PASS T03 no TEST key is read')
TR = build('TRAIN'); VA = build('VAL'); CA = build('CAL', only_replica0=True)
for nm_, D in [('TRAIN', TR), ('VAL', VA), ('CAL', CA)]: say(f'{nm_}: ' + str({k: dict(zip(*np.unique(v[1], return_counts=True))) for k, v in D.items() if k == 'ORACLE'}))

PASS T01 signature uses only the episode bars (later bars cannot change it)
PASS T02 jitter is deterministic per event and never starts before the event
PASS T03 no TEST key is read
TRAIN: {'ORACLE': {np.int64(1): np.int64(200), np.int64(2): np.int64(200), np.int64(3): np.int64(182), np.int64(4): np.int64(200), np.int64(5): np.int64(200), np.int64(6): np.int64(200)}}
VAL: {'ORACLE': {np.int64(1): np.int64(76), np.int64(2): np.int64(76), np.int64(3): np.int64(72), np.int64(4): np.int64(76), np.int64(5): np.int64(76), np.int64(6): np.int64(76), np.int64(7): np.int64(72), np.int64(8): np.int64(72)}}
CAL: {'ORACLE': {np.int64(1): np.int64(50), np.int64(2): np.int64(50), np.int64(3): np.int64(44), np.int64(4): np.int64(50), np.int64(5): np.int64(50), np.int64(6): np.int64(50), np.int64(7): np.int64(50), np.int64(8): np.int64(50)}}


## 4 · Fit, tune, evaluate (CAL-3 once)

In [ ]:
# fit on TRAIN (known families only), tune on VAL (known only), evaluate CAL once
def sel(D, fams): X, y, u, s = D; m = np.isin(y, fams); return X[m], y[m], u[m], s[m]
Xtr, ytr, _, _ = sel(TR['ORACLE'], KNOWN); Xva, yva, _, _ = sel(VA['ORACLE'], KNOWN)
assert not np.isin(ytr, UNSEEN).any() and not np.isin(yva, UNSEEN).any(); assert len(set(ytr)) == 6 and len(set(yva)) == 6
sc, lr, C, cent, P = fit_known(Xtr, ytr, Xva, yva); thr = thresholds(sc, cent, P, Xva, yva)
say(f'chosen C = {C}; reject thresholds (90th pct of VAL own-centroid distance): ' + str({FAMN[f]: round(v, 2) for f, v in thr.items()}))
RES = {}
for var in ['ORACLE', 'JITTERED']:
    Xk, yk, _, sk = sel(CA[var], KNOWN); Xu, yu, _, _ = sel(CA[var], UNSEEN)
    Pr, top, d, rej = predict(sc, lr, cent, P, thr, Xk); h1 = topk_hits(Pr, lr.classes_, yk, 1); h2 = topk_hits(Pr, lr.classes_, yk, 2)
    _, topu, du, reju = predict(sc, lr, cent, P, thr, Xu)
    perf = {FAMN[f]: float(h1[yk == f].mean()) for f in KNOWN}
    # plain-loop recomputation of macro top-1
    loop = float(np.mean([np.mean([1.0 if top[i] == yk[i] else 0.0 for i in range(len(yk)) if yk[i] == f]) for f in KNOWN])); assert abs(loop - macro(yk, h1)) < 1e-12
    conf = pd.crosstab(pd.Series([FAMN[v] for v in yk], name='true'), pd.Series([FAMN[v] for v in top], name='predicted'))
    RES[var] = {'top1': macro(yk, h1), 'top1_lo': boot_lower(yk, h1), 'top2': macro(yk, h2), 'known_reject': macro(yk, rej.astype(float)), 'unseen_reject': macro(yu, reju.astype(float)),
                'unseen_reject_by_family': {FAMN[f]: float(reju[yu == f].mean()) for f in UNSEEN}, 'per_family_top1': perf, 'n_known': int(len(yk)), 'n_unseen': int(len(yu)),
                'unseen_nearest': {FAMN[f]: {FAMN[k]: int((topu[yu == f] == k).sum()) for k in KNOWN} for f in UNSEEN}, 'conf': conf, 'per_symbol_top1': {s_: macro(yk[sk == s_], h1[sk == s_]) for s_ in SYMBOLS}}
    say(''); say(f'--- {var} boundaries (CAL-3, replica 0; {len(yk)} known events, {len(yu)} unseen events) ---')
    say(f"macro top-1 {RES[var]['top1']:.3f} (bootstrap lower {RES[var]['top1_lo']:.3f}), macro top-2 {RES[var]['top2']:.3f}, known reject {RES[var]['known_reject']:.3f}, unseen reject {RES[var]['unseen_reject']:.3f}")
    say('per family top-1: ' + str({k: round(v, 2) for k, v in perf.items()})); say('unseen reject by family: ' + str({k: round(v, 2) for k, v in RES[var]['unseen_reject_by_family'].items()}))
    say('per symbol top-1: ' + str({k: round(v, 3) for k, v in RES[var]['per_symbol_top1'].items()})); say('confusion (rows true, columns predicted):'); say(conf.to_string())
    say('unseen events: nearest known family counts: ' + str(RES[var]['unseen_nearest']))

chosen C = 10; reject thresholds (90th pct of VAL own-centroid distance): {'pump_and_dump_like': 5.1, 'fake_breakout_like': 5.32, 'abnormal_volume_burst': 6.19, 'trade_flow_imbalance': 6.04, 'wash_like_aggregate_proxy': 5.66, 'liquidity_stress_proxy': 6.57}

--- ORACLE boundaries (CAL-3, replica 0; 294 known events, 100 unseen events) ---
macro top-1 0.738 (bootstrap lower 0.693), macro top-2 0.911, known reject 0.176, unseen reject 0.180
per family top-1: {'pump_and_dump_like': 0.6, 'fake_breakout_like': 0.54, 'abnormal_volume_burst': 0.73, 'trade_flow_imbalance': 0.56, 'wash_like_aggregate_proxy': 1.0, 'liquidity_stress_proxy': 1.0}
unseen reject by family: {'momentum_ignition_like': 0.3, 'slow_accumulation_like': 0.06}
per symbol top-1: {'BTCUSDT': 0.775, 'ETHUSDT': 0.701}
confusion (rows true, columns predicted):
predicted                  abnormal_volume_burst  fake_breakout_like  liquidity_stress_proxy  pump_and_dump_like  trade_flow_imbalance  wash_like_aggregate_proxy
true     

## 5 · Hypotheses and product rule

In [ ]:
O, J = RES['ORACLE'], RES['JITTERED']
HG1 = bool(O['top1'] >= 0.50 and O['top1_lo'] > 0.33); HG2 = bool(O['top2'] >= 0.75); HG3 = bool(J['unseen_reject'] >= 0.40 and J['known_reject'] <= 0.15); HG5 = bool(J['top1'] >= 0.40 and J['top2'] >= 0.65)
say(''); say(f'H-G1 (ORACLE top-1 >= 0.50, bootstrap lower > 0.33): {HG1}'); say(f'H-G2 (ORACLE top-2 >= 0.75): {HG2}'); say(f'H-G3 (JITTERED unseen reject >= 0.40 and known reject <= 0.15): {HG3}'); say(f'H-G5 (JITTERED top-1 >= 0.40 and top-2 >= 0.65): {HG5}')
SHOW_CLOSEST = bool(HG1 and HG2 and HG5); SHOW_NOMATCH = bool(HG3)
say(f'PRODUCT RULE -> show "Resembles: X (indicative)": {SHOW_CLOSEST}; show "No close match": {SHOW_NOMATCH}')
if not SHOW_CLOSEST: say('The closest-pattern output stays HIDDEN on the dashboard; report the failure honestly.')
if not SHOW_NOMATCH: say('The no-close-match output stays HIDDEN on the dashboard; report the failure honestly.')


H-G1 (ORACLE top-1 >= 0.50, bootstrap lower > 0.33): True
H-G2 (ORACLE top-2 >= 0.75): True
H-G3 (JITTERED unseen reject >= 0.40 and known reject <= 0.15): False
H-G5 (JITTERED top-1 >= 0.40 and top-2 >= 0.65): True
PRODUCT RULE -> show "Resembles: X (indicative)": True; show "No close match": False
The no-close-match output stays HIDDEN on the dashboard; report the failure honestly.


## 6 · Save and declare

In [ ]:
PLAIN = {1: 'Pump-and-dump-like surge', 2: 'Fake-breakout-like move', 3: 'Abnormal volume burst', 4: 'One-sided trade flow', 5: 'Wash-like trading (proxy)', 6: 'Liquidity-stress-like (proxy)'}
MODEL = {'status': 'DECLARED: pattern-resemblance layer (protocol G1 v1). Read-only on saved features.', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
         'protocol_g1_sha16': gh[:16], 'normalisation': MODE, 'features': FEATS, 'signature': 'per feature max|z| (6), signed mean (6), log duration (1)', 'C': C, 'classes': [int(c) for c in lr.classes_],
         'plain_names': {str(k): v for k, v in PLAIN.items()}, 'scaler_mean': sc.mean_.tolist(), 'scaler_scale': sc.scale_.tolist(), 'coef': lr.coef_.tolist(), 'intercept': lr.intercept_.tolist(),
         'centroids': {str(k): v.tolist() for k, v in cent.items()}, 'precision': P.tolist(), 'reject_threshold': {str(k): v for k, v in thr.items()},
         'hypotheses': {'H_G1': HG1, 'H_G2': HG2, 'H_G3': HG3, 'H_G5': HG5}, 'show_closest': SHOW_CLOSEST, 'show_no_match': SHOW_NOMATCH,
         'results': {v: {k: x for k, x in RES[v].items() if k != 'conf'} for v in RES}}
(OUT/'declared_pattern_resemblance_v1.json').write_text(json.dumps(MODEL, indent=1))
for v in RES: RES[v]['conf'].to_csv(OUT/f'pattern_resemblance_confusion_{v.lower()}_cal3.csv')
block = '\n'.join(LINES); (OUT/'b2_step_g1_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step_g1_share_block.txt, or copy everything below) ====='); print(block); print('saved', OUT/'declared_pattern_resemblance_v1.json')


===== SHARE BLOCK (attach models_b2_v1/b2_step_g1_share_block.txt, or copy everything below) =====
protocol G1 frozen now, sha256 6ffd7e485c67f253
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944
normalisation mode N0; families: {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy', 6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}
splits found in layer3_v3: {'BTCUSDT': ['CAL', 'TEST', 'TRAIN', 'VAL'], 'ETHUSDT': ['CAL', 'TEST', 'TRAIN', 'VAL']}
using splits: {'BTCUSDT': {'TRAIN': 'TRAIN', 'VAL': 'VAL', 'CAL': 'CAL'}, 'ETHUSDT': {'TRAIN': 'TRAIN', 'VAL': 'VAL', 'CAL': 'CAL'}}
PASS T01 signature uses only the episode bars (later bars cannot change it)
PASS T02 jitter is deterministic per event and never starts before the event
PASS T03 no TEST key is read
TRAIN: {'ORACLE': {np.int64(1): np.int64(200), np.int64(2): np.int64(200), np.int64(3)